# Gas-Water (Air-Water) Generalization Check -- Sister Notebook

Companion to `mixtureTheoryFlow_jax.ipynb`, split off 2026-10-28 to keep
the oil-water and gas-water work separate. See CLAUDE.md's "Reduced-Model
Water-Gas Solver" section for the full reasoning and current status.


In [ ]:
# Reduced (alpha_l, Delta) solver -- different state/API than the main
# notebook's solver. See CLAUDE.md's "Reduced-Model Water-Gas Solver".
from mixtureTheoryFlow_jax_solver_water_gas import *


# Carraretto et al. (2020), MDPI -- Zero-Refit Run (Reduced-Model Solver)

Air/water stratified flow, 24m/60mm PMMA pipe, digitized from Figures 6,
9, 10 (`mdpi_airwater_phase2g_dataset.csv`). Horizontal only (`beta=0.0`),
uncalibrated (`f_i_mult=1.0`). See CLAUDE.md's "Reduced-Model Water-Gas
Solver" section for why this dataset motivated the new solver.


## Load the combined MDPI Phase 2g dataset (Figures 6/9/10, digitized)

In [ ]:
# Carraretto et al. (2020), digitized Figures 6/9/10. JL/JG are already
# U_sl/U_sg (liquid/gas superficial velocity) -- just renamed below.

import pandas as pd

mdpi_df = pd.read_csv('mdpi_airwater_phase2g_dataset.csv')
mdpi_clean = mdpi_df[['Um_target', 'WC', 'dpdz_measured', 'psi', 'flow_regime', 'JL', 'JG']].copy()
mdpi_clean = mdpi_clean.rename(columns={'JL': 'U_sl', 'JG': 'U_sg'})

print(f"Loaded {len(mdpi_clean)} conditions from Carraretto et al. (2020)")
print(f"U_sl range: {mdpi_clean['U_sl'].min():.4f}-{mdpi_clean['U_sl'].max():.4f} m/s")
print(mdpi_clean.to_string())


## Validate solver inputs before running


In [ ]:
# D/rho_l/rho_g/mu_l/mu_g/L are fixed constants (next cell) -- only
# U_sl/U_sg need checking here.

required_cols = ['U_sl', 'U_sg']
invalid = mdpi_clean[required_cols].isna().any(axis=1) | mdpi_clean[required_cols].le(0).any(axis=1)
n_bad = int(invalid.sum())
if n_bad:
    print(f"WARNING: {n_bad}/{len(mdpi_clean)} rows have invalid U_sl/U_sg -- see below")
    print(mdpi_clean[invalid])
else:
    print(f"All {len(mdpi_clean)} rows have finite, positive U_sl/U_sg -- safe to run.")


## Run every MDPI condition to steady state, zero-refit

No oscillation sweep/quiet-window step needed -- see CLAUDE.md.


In [ ]:
from mixtureTheoryFlow_jax_solver_water_gas import run_real_data_condition
import numpy as np
import pickle

D_mdpi     = 0.060      # pipe diameter [m] -- 60mm ID
L_mdpi     = 24.0       # pipe length [m] -- full 24m PMMA test section
rho_l_mdpi = 998.0      # water [kg/m^3]
rho_g_mdpi = 1.204      # air [kg/m^3] -- 20C, 1 atm
mu_l_mdpi  = 1.0e-3     # water [Pa.s] -- 20C
mu_g_mdpi  = 1.81e-5    # air [Pa.s] -- 20C, 1 atm

mdpi_results = []
for i, row in mdpi_clean.iterrows():
    res = run_real_data_condition(
        row['U_sl'], row['U_sg'], D_mdpi, rho_l_mdpi, rho_g_mdpi, mu_l_mdpi, mu_g_mdpi, L_mdpi
    )
    res['i'] = i
    res['alpha_l_meas'] = row['psi']
    res['dpdz_meas']    = row['dpdz_measured']
    mdpi_results.append(res)
    print(f"[{i:2d}] U_sl={row['U_sl']:.4f}  U_sg={row['U_sg']:.3f}  "
          f"alpha_l_pred={res['alpha_l_pred']:.4f} (meas {row['psi']:.4f})  "
          f"dPdz_l_pred={res['dPdz_pred_liquid']:.2f} (meas {row['dpdz_measured']:.2f})  "
          f"kh={res['kh_total']}  any_nan={res['any_nan']}")

# kh_total>0 means hyperbolicity broke down mid-run (see CLAUDE.md) -- the
# output can look plausible but isn't trustworthy, so exclude before any
# error stat or future calibration fit, same treatment PHI_COLLAPSE got
# in the old solver.
stable = [r for r in mdpi_results if r['kh_total'] == 0]
n_kh_unstable = len(mdpi_results) - len(stable)
if n_kh_unstable:
    print(f"\nWARNING: {n_kh_unstable}/{len(mdpi_results)} conditions had kh_total>0 -- "
          f"excluded from error stats below, not trustworthy.")

alpha_err = np.array([abs(r['alpha_l_pred'] - r['alpha_l_meas']) / r['alpha_l_meas'] for r in stable])
dpdz_err  = np.array([(abs(r['dPdz_pred_liquid']) - r['dpdz_meas']) / r['dpdz_meas'] for r in stable])
print(f"\nMDPI zero-refit (f_i_mult=1.0), {len(stable)}/{len(mdpi_results)} kh-stable: "
      f"holdup mean|err|={alpha_err.mean():.1%}  "
      f"dPdz mean(signed err)={dpdz_err.mean():.1%}  "
      f"any_nan={sum(r['any_nan'] for r in mdpi_results)}/{len(mdpi_results)}")

with open('mdpi_real_data_results.pkl', 'wb') as f:
    pickle.dump(mdpi_results, f)
print("Saved to mdpi_real_data_results.pkl")


## Fan, Y. (2005) -- Low Liquid Loading Stratified-Wavy (horizontal subset)

`fan2005_sw_subset.csv`, filtered to `Authors=="Fan, Y."`,
`Flow Pattern=="SW"`, 13 horizontal rows (`Inclination angle == 0`) of 60
total -- see CLAUDE.md for why the other 47 stay excluded for now.


### Load Fan, Y. (2005) stratified-wavy (SW) subset

In [3]:
# Fan, Y. (2005), SW subset, horizontal rows only -- beta=0 is the only
# validated regime so far (see CLAUDE.md).

import pandas as pd

fan2005_df = pd.read_csv('fan2005_sw_subset.csv')
fan2005_horiz = fan2005_df[fan2005_df['Inclination angle[°]'] == 0].copy()

fan2005_clean = fan2005_horiz.rename(columns={
    'vSL[m/s]': 'U_sl', 'vSG[m/s]': 'U_sg',
    'rhoL[kg/m3]': 'rho_l', 'rhoG[kg/m3]': 'rho_g',
    'viscosityL[Pa.s]': 'mu_l', 'viscosityG[Pa.s]': 'mu_g',
    'Pipe Diameter[m]': 'D', 'Pipe Length[m]': 'L',
})[['U_sl', 'U_sg', 'D', 'rho_l', 'rho_g', 'mu_l', 'mu_g', 'L']].copy()

fan2005_clean['psi']           = pd.to_numeric(fan2005_horiz['HL[-]'], errors='coerce')  # '-' -> NaN
fan2005_clean['dpdz_measured'] = fan2005_horiz['dP/dL[Pa/m]']
fan2005_clean = fan2005_clean.reset_index(drop=True)

n_no_psi = fan2005_clean['psi'].isna().sum()
print(f"Loaded {len(fan2005_clean)} horizontal SW conditions from Fan, Y. (2005)")
print(f"  {n_no_psi}/{len(fan2005_clean)} have no holdup (HL) measurement -- dP/dL-only")
print(fan2005_clean.to_string())


Loaded 13 horizontal SW conditions from Fan, Y. (2005)
  3/13 have no holdup (HL) measurement -- excluded from holdup checks, Um-only
  WC range: 5.20e-05 - 1.23e-03  (PHI_COLLAPSE=0.1 -- every condition sits far below this)
    Um_target        WC  dpdz_measured   psi flow_regime  rho2_val
0   15.128551  0.001226         202.08  0.02          SW      2.57
1   19.724320  0.000726         352.72  0.02          SW      2.67
2   15.103639  0.000903         191.78  0.02          SW      2.57
3   15.130954  0.000724         173.70  0.02          SW      2.57
4   19.890300  0.000518         300.24  0.01          SW      2.66
5   14.937766  0.000520         183.17  0.01          SW      2.56
6   19.866065  0.000305         269.21  0.01          SW      2.65
7   14.974530  0.000303         165.56  0.01          SW      2.55
8   20.263120  0.000154         247.31  0.01          SW      2.52
9   14.852289  0.000154         118.79  0.01          SW      2.43
10  24.961297  0.000052         294.20

### Validate solver inputs before running


In [ ]:
# This dataset has known placeholder gaps elsewhere (see load cell) --
# check the solver-input columns specifically here.

required_cols = ['U_sl', 'U_sg', 'D', 'rho_l', 'rho_g', 'mu_l', 'mu_g', 'L']
invalid = fan2005_clean[required_cols].isna().any(axis=1) | fan2005_clean[required_cols].le(0).any(axis=1)
n_bad = int(invalid.sum())
if n_bad:
    print(f"WARNING: {n_bad}/{len(fan2005_clean)} rows have invalid solver inputs -- see below")
    print(fan2005_clean[invalid])
else:
    print(f"All {len(fan2005_clean)} rows have finite, positive solver inputs -- safe to run.")


### Run every horizontal Fan (2005) condition to steady state, zero-refit


In [7]:
from mixtureTheoryFlow_jax_solver_water_gas import run_real_data_condition
import numpy as np
import pickle

fan2005_results = []
for i, row in fan2005_clean.iterrows():
    res = run_real_data_condition(
        row['U_sl'], row['U_sg'], row['D'], row['rho_l'], row['rho_g'],
        row['mu_l'], row['mu_g'], row['L']
    )
    res['i'] = i
    res['alpha_l_meas'] = row['psi']
    res['dpdz_meas']    = row['dpdz_measured']
    fan2005_results.append(res)
    psi_str = f"{row['psi']:.4f}" if pd.notna(row['psi']) else "n/a"
    print(f"[{i:2d}] U_sl={row['U_sl']:.6f}  U_sg={row['U_sg']:.3f}  "
          f"alpha_l_pred={res['alpha_l_pred']:.6f} (meas {psi_str})  "
          f"dPdz_l_pred={res['dPdz_pred_liquid']:.2f} (meas {row['dpdz_measured']:.2f})  "
          f"kh={res['kh_total']}  any_nan={res['any_nan']}")

# kh_total>0 means hyperbolicity broke down mid-run (see CLAUDE.md) -- the
# output can look plausible but isn't trustworthy, so exclude before any
# error stat or future calibration fit, same treatment PHI_COLLAPSE got
# in the old solver.
stable = [r for r in fan2005_results if r['kh_total'] == 0]
n_kh_unstable = len(fan2005_results) - len(stable)
if n_kh_unstable:
    print(f"\nWARNING: {n_kh_unstable}/{len(fan2005_results)} conditions had kh_total>0 -- "
          f"excluded from error stats below, not trustworthy.")

have_psi  = [r for r in stable if pd.notna(r['alpha_l_meas'])]
alpha_err = np.array([abs(r['alpha_l_pred'] - r['alpha_l_meas']) / r['alpha_l_meas'] for r in have_psi])
dpdz_err  = np.array([(abs(r['dPdz_pred_liquid']) - r['dpdz_meas']) / r['dpdz_meas'] for r in stable])

print(f"\nFan (2005) horizontal, zero-refit (f_i_mult=1.0), {len(stable)}/{len(fan2005_results)} kh-stable: "
      f"holdup mean|err|={alpha_err.mean():.1%} ({len(have_psi)}/{len(stable)} have HL)  "
      f"dPdz mean(signed err)={dpdz_err.mean():.1%}  "
      f"any_nan={sum(r['any_nan'] for r in fan2005_results)}/{len(fan2005_results)}")

with open('fan2005_real_data_results.pkl', 'wb') as f:
    pickle.dump(fan2005_results, f)
print("Saved to fan2005_real_data_results.pkl")



Condition: WC=0.001226  Um=15.13  dpdz=202.08  psi=0.02  rho2=2.570
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 2.0637 (target 15.13)

Condition: WC=0.000726  Um=19.72  dpdz=352.72  psi=0.02  rho2=2.670
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 2.1416 (target 19.72)

Condition: WC=0.000903  Um=15.10  dpdz=191.78  psi=0.02  rho2=2.570
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 2.0067 (target 15.10)

Condition: WC=0.000724  Um=15.13  dpdz=173.70  psi=0.02  rho2=2.570
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 1.8931 (target 15.13)

Condition: WC=0.000518  Um=19.89  dpdz=300.24  psi=0.01  rho2=2.660
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 1.9403 (target 19.89)

Condition: WC=0.000520  Um=14.94  dpdz=183.17  psi=0.01  rho2=2.560
  peak phi2_range = 0.0000 at t=0.7s   flag(>0.3) = False   Um_sim@end = 1.9802 (target 14.94)

Condition: WC=0